In [1]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
!pip install mordred --no-index --find-links=file:///kaggle/input/mordred-1-2-0-py3-none-any/
!pip install torch_geometric --no-index --find-links=file:/kaggle/input/torch-geometric-2-6-1-whl/torch_geometric-2.6.1-py3-none-any.whl

Processing /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
Looking in links: file:///kaggle/input/mordred-1-2-0-py3-none-any/
Processing /kaggle/input/mordred-1-2-0-py3-none-any/mordred-1.2.0-py3-none-any.whl
Processing /kaggle/input/mordred-1-2-0-py3-none-any/networkx-2.8.8-py3-none-any.whl (from mordred)
  Attempting uninstall: networkx
    Found existing installation: networkx 3.5
    Uninstalling networkx-3.5:
      Successfully uninstalled networkx-3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import (GCNConv, GATConv, GraphConv, TransformerConv, 
                               global_mean_pool, global_max_pool, global_add_pool, 
                               BatchNorm, LayerNorm)
from torch_geometric.data import Data, Batch
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from collections import defaultdict
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, Descriptors, Crippen, Lipinski, AllChem
from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect
import networkx as nx

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
train = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')
test = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

class AdvancedSMILESToGraph:
    """Advanced SMILES to molecular graph converter with comprehensive features"""
    
    def __init__(self):
        # Comprehensive atom vocabulary including metals and rare elements
        self.atom_types = [
            'C', 'N', 'O', 'S', 'P', 'F', 'Cl', 'Br', 'I', 'H', 
            'B', 'Si', 'Se', 'Te', 'As', 'Na', 'K', 'Li', 'Mg', 
            'Ca', 'Al', 'Zn', 'Fe', 'Cu', 'Mn', 'Ni', 'Co', 'Ti',
            'V', 'Cr', 'Sn', 'Pb', 'Ag', 'Au', 'Pd', 'Pt'
        ]
        self.atom_to_idx = {atom: idx for idx, atom in enumerate(self.atom_types)}
        self.atom_to_idx['UNK'] = len(self.atom_types)
        
        # Extended hybridization types
        self.hybridization_types = [
            Chem.rdchem.HybridizationType.SP,
            Chem.rdchem.HybridizationType.SP2, 
            Chem.rdchem.HybridizationType.SP3,
            Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2,
            Chem.rdchem.HybridizationType.UNSPECIFIED
        ]
        
        # Comprehensive bond types
        self.bond_types = [
            Chem.rdchem.BondType.SINGLE,
            Chem.rdchem.BondType.DOUBLE,
            Chem.rdchem.BondType.TRIPLE,
            Chem.rdchem.BondType.AROMATIC,
            Chem.rdchem.BondType.DATIVEONE,
            Chem.rdchem.BondType.DATIVEL,
            Chem.rdchem.BondType.DATIVER
        ]
        
        # Extended atomic properties
        self.electronegativity = {
            'C': 2.55, 'N': 3.04, 'O': 3.44, 'S': 2.58, 'P': 2.19, 'F': 3.98,
            'Cl': 3.16, 'Br': 2.96, 'I': 2.66, 'H': 2.20, 'B': 2.04, 'Si': 1.90,
            'Se': 2.55, 'Te': 2.1, 'As': 2.18, 'Na': 0.93, 'K': 0.82, 'Li': 0.98,
            'Mg': 1.31, 'Ca': 1.00, 'Al': 1.61, 'Zn': 1.65, 'Fe': 1.83,
            'Cu': 1.90, 'Mn': 1.55, 'Ni': 1.91, 'Co': 1.88, 'Ti': 1.54,
            'V': 1.63, 'Cr': 1.66, 'Sn': 1.96, 'Pb': 2.33, 'Ag': 1.93,
            'Au': 2.54, 'Pd': 2.20, 'Pt': 2.28
        }
        
        self.atomic_radius = {
            'C': 0.76, 'N': 0.71, 'O': 0.66, 'S': 1.05, 'P': 1.07, 'F': 0.57,
            'Cl': 0.99, 'Br': 1.14, 'I': 1.33, 'H': 0.31, 'B': 0.84, 'Si': 1.11,
            'Se': 1.20, 'Te': 1.38, 'As': 1.19, 'Na': 1.66, 'K': 2.03, 'Li': 1.28,
            'Mg': 1.41, 'Ca': 1.76, 'Al': 1.21, 'Zn': 1.22, 'Fe': 1.52,
            'Cu': 1.32, 'Mn': 1.61, 'Ni': 1.24, 'Co': 1.26, 'Ti': 1.60,
            'V': 1.53, 'Cr': 1.39, 'Sn': 1.39, 'Pb': 1.54, 'Ag': 1.45,
            'Au': 1.36, 'Pd': 1.39, 'Pt': 1.36
        }
        
        # Ionization energies (normalized by 1000)
        self.ionization_energy = {
            'C': 11.26, 'N': 14.53, 'O': 13.62, 'S': 10.36, 'P': 10.49, 'F': 17.42,
            'Cl': 12.97, 'Br': 11.81, 'I': 10.45, 'H': 13.60, 'B': 8.30, 'Si': 8.15,
            'Se': 9.75, 'Te': 9.01, 'As': 9.79, 'Na': 5.14, 'K': 4.34, 'Li': 5.39,
            'Mg': 7.65, 'Ca': 6.11, 'Al': 5.99, 'Zn': 9.39, 'Fe': 7.90
        }
    
    def get_atom_features(self, atom):
        """Extract comprehensive atom features"""
        try:
            features = []
            atom_symbol = atom.GetSymbol()
            
            # One-hot encoding for atom type
            atom_onehot = [0] * (len(self.atom_types) + 1)
            if atom_symbol in self.atom_to_idx:
                atom_onehot[self.atom_to_idx[atom_symbol]] = 1
            else:
                atom_onehot[self.atom_to_idx['UNK']] = 1
            features.extend(atom_onehot)
            
            # Degree (one-hot encoding for 0-10)
            degree = atom.GetDegree()
            degree_onehot = [0] * 11
            if degree <= 10:
                degree_onehot[degree] = 1
            features.extend(degree_onehot)
            
            # Total degree (including implicit hydrogens)
            total_degree = atom.GetTotalDegree()
            total_degree_onehot = [0] * 11
            if total_degree <= 10:
                total_degree_onehot[total_degree] = 1
            features.extend(total_degree_onehot)
            
            # Formal charge (one-hot for common charges: -3 to +3)
            formal_charge = atom.GetFormalCharge()
            charge_onehot = [0] * 7  # -3, -2, -1, 0, +1, +2, +3
            if -3 <= formal_charge <= 3:
                charge_onehot[formal_charge + 3] = 1
            features.extend(charge_onehot)
            
            # Hybridization (one-hot)
            try:
                hybridization = atom.GetHybridization()
                hyb_onehot = [0] * len(self.hybridization_types)
                if hybridization in self.hybridization_types:
                    hyb_onehot[self.hybridization_types.index(hybridization)] = 1
                features.extend(hyb_onehot)
            except:
                features.extend([0] * len(self.hybridization_types))
            
            # Boolean features
            features.append(int(atom.GetIsAromatic()))
            features.append(int(atom.IsInRing()))
            features.append(int(atom.IsInRingSize(3)))
            features.append(int(atom.IsInRingSize(4)))
            features.append(int(atom.IsInRingSize(5)))
            features.append(int(atom.IsInRingSize(6)))
            features.append(int(atom.IsInRingSize(7)))
            features.append(int(atom.IsInRingSize(8)))
            
            # Number of implicit and explicit hydrogens
            features.append(atom.GetTotalNumHs())
            features.append(atom.GetNumImplicitHs())
            features.append(atom.GetNumExplicitHs())
            
            # Valence information
            try:
                features.append(atom.GetTotalValence())
                features.append(atom.GetNumRadicalElectrons())
            except:
                features.extend([0, 0])
            
            # Chirality
            try:
                chiral_tag = atom.GetChiralTag()
                chiral_onehot = [0] * 4  # Unspecified, R, S, Other
                if chiral_tag == Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW:
                    chiral_onehot[1] = 1
                elif chiral_tag == Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW:
                    chiral_onehot[2] = 1
                elif chiral_tag != Chem.rdchem.ChiralType.CHI_UNSPECIFIED:
                    chiral_onehot[3] = 1
                else:
                    chiral_onehot[0] = 1
                features.extend(chiral_onehot)
            except:
                features.extend([1, 0, 0, 0])
            
            # Atomic properties (normalized)
            electronegativity = self.electronegativity.get(atom_symbol, 2.5) / 4.0
            features.append(electronegativity)
            
            atomic_radius = self.atomic_radius.get(atom_symbol, 1.0) / 2.5
            features.append(atomic_radius)
            
            ionization_energy = self.ionization_energy.get(atom_symbol, 10.0) / 20.0
            features.append(ionization_energy)
            
            # Mass (normalized)
            mass = atom.GetMass() / 200.0
            features.append(mass)
            
            # Atomic number (normalized)
            atomic_num = atom.GetAtomicNum() / 118.0
            features.append(atomic_num)
            
            return features
            
        except Exception as e:
            print(f"Error in get_atom_features: {e}")
            # Return default features with correct dimension
            return [1] + [0] * 82  # Adjust based on actual feature count
    
    def get_bond_features(self, bond):
        """Extract comprehensive bond features"""
        try:
            features = []
            
            # Bond type one-hot
            bond_type = bond.GetBondType()
            bond_onehot = [0] * len(self.bond_types)
            if bond_type in self.bond_types:
                bond_onehot[self.bond_types.index(bond_type)] = 1
            else:
                bond_onehot[0] = 1  # Default to single
            features.extend(bond_onehot)
            
            # Boolean features
            features.append(int(bond.GetIsConjugated()))
            features.append(int(bond.IsInRing()))
            features.append(int(bond.IsInRingSize(3)))
            features.append(int(bond.IsInRingSize(4)))
            features.append(int(bond.IsInRingSize(5)))
            features.append(int(bond.IsInRingSize(6)))
            features.append(int(bond.IsInRingSize(7)))
            
            # Stereo
            try:
                stereo = bond.GetStereo()
                stereo_onehot = [0] * 4  # None, E, Z, Other
                if stereo == Chem.rdchem.BondStereo.STEREOE:
                    stereo_onehot[1] = 1
                elif stereo == Chem.rdchem.BondStereo.STEREOZ:
                    stereo_onehot[2] = 1
                elif stereo != Chem.rdchem.BondStereo.STEREONONE:
                    stereo_onehot[3] = 1
                else:
                    stereo_onehot[0] = 1
                features.extend(stereo_onehot)
            except:
                features.extend([1, 0, 0, 0])
            
            return features
            
        except Exception as e:
            print(f"Error in get_bond_features: {e}")
            return [1] + [0] * 17  # Default bond features
    
    def get_global_features(self, mol, smiles):
        """Extract comprehensive global molecular features"""
        try:
            features = []
            
            # Basic descriptors
            features.append(mol.GetNumAtoms() / 100.0)  # Normalized
            features.append(mol.GetNumBonds() / 150.0)
            features.append(mol.GetNumHeavyAtoms() / 80.0)
            
            # Ring descriptors
            features.append(rdMolDescriptors.CalcNumRings(mol) / 10.0)
            features.append(rdMolDescriptors.CalcNumAromaticRings(mol) / 5.0)
            features.append(rdMolDescriptors.CalcNumSaturatedRings(mol) / 5.0)
            features.append(rdMolDescriptors.CalcNumHeterocycles(mol) / 5.0)
            
            # Molecular weight and size
            features.append(Descriptors.MolWt(mol) / 1000.0)
            features.append(Descriptors.ExactMolWt(mol) / 1000.0)
            
            # Lipophilicity
            try:
                features.append(Crippen.MolLogP(mol))
                features.append(Crippen.MolMR(mol) / 100.0)  # Molar refractivity
            except:
                features.extend([0.0, 0.0])
            
            # Surface area and volume
            try:
                features.append(rdMolDescriptors.CalcTPSA(mol) / 200.0)
                features.append(rdMolDescriptors.CalcLabuteASA(mol) / 500.0)
            except:
                features.extend([0.0, 0.0])
            
            # H-bonding
            features.append(Lipinski.NumHDonors(mol) / 10.0)
            features.append(Lipinski.NumHAcceptors(mol) / 10.0)
            features.append(Lipinski.NumRotatableBonds(mol) / 20.0)
            
            # Flexibility and shape
            try:
                features.append(rdMolDescriptors.CalcFractionCsp3(mol))
                features.append(rdMolDescriptors.CalcKappa1(mol) / 20.0)
                features.append(rdMolDescriptors.CalcKappa2(mol) / 10.0)
                features.append(rdMolDescriptors.CalcKappa3(mol) / 5.0)
            except:
                features.extend([0.0, 0.0, 0.0, 0.0])
            
            # Electronic descriptors
            try:
                features.append(rdMolDescriptors.CalcNumHBA(mol) / 10.0)
                features.append(rdMolDescriptors.CalcNumHBD(mol) / 10.0)
            except:
                features.extend([0.0, 0.0])
            
            # Complexity descriptors
            try:
                features.append(rdMolDescriptors.BertzCT(mol) / 1000.0)
            except:
                features.append(0.0)
            
            # Additional pharmacophore features
            try:
                features.append(len([x for x in mol.GetAtoms() if x.GetSymbol() == 'N']) / 10.0)  # N count
                features.append(len([x for x in mol.GetAtoms() if x.GetSymbol() == 'O']) / 10.0)  # O count
                features.append(len([x for x in mol.GetAtoms() if x.GetSymbol() == 'S']) / 5.0)   # S count
                features.append(len([x for x in mol.GetAtoms() if x.GetSymbol() == 'P']) / 5.0)   # P count
                features.append(len([x for x in mol.GetAtoms() if x.GetSymbol() in ['F', 'Cl', 'Br', 'I']]) / 5.0)  # Halogen count
            except:
                features.extend([0.0, 0.0, 0.0, 0.0, 0.0])
            
            return torch.tensor(features, dtype=torch.float)
            
        except Exception as e:
            print(f"Error in global features: {e}")
            return torch.zeros(30, dtype=torch.float)  # Adjusted size
    
    def smiles_to_graph(self, smiles):
        """Convert SMILES to molecular graph with enhanced features"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                mol = Chem.MolFromSmiles(smiles, sanitize=False)
                if mol is None:
                    return self.create_dummy_graph()
                try:
                    Chem.SanitizeMol(mol)
                except:
                    return self.create_dummy_graph()
            
            # Add hydrogens for better feature extraction
            mol = Chem.AddHs(mol)
            
            # Get atom features
            atom_features = []
            for atom in mol.GetAtoms():
                atom_feats = self.get_atom_features(atom)
                atom_features.append(atom_feats)
            
            if len(atom_features) == 0:
                return self.create_dummy_graph()
            
            # Ensure consistent feature length
            max_feat_len = max(len(feat) for feat in atom_features)
            for i in range(len(atom_features)):
                while len(atom_features[i]) < max_feat_len:
                    atom_features[i].append(0.0)
                atom_features[i] = atom_features[i][:max_feat_len]  # Truncate if too long
            
            node_features = torch.tensor(atom_features, dtype=torch.float)
            
            # Get edges and edge features
            edges = []
            edge_features = []
            
            for bond in mol.GetBonds():
                i = bond.GetBeginAtomIdx()
                j = bond.GetEndAtomIdx()
                edge_feat = self.get_bond_features(bond)
                
                # Add both directions for undirected graph
                edges.extend([[i, j], [j, i]])
                edge_features.extend([edge_feat, edge_feat])
            
            if len(edges) == 0:
                # Single atom molecule
                edges = [[0, 0]]
                edge_features = [self.get_bond_features(None)]  # Dummy bond
            
            # Ensure consistent edge feature length
            if edge_features:
                max_edge_len = max(len(feat) for feat in edge_features)
                for i in range(len(edge_features)):
                    while len(edge_features[i]) < max_edge_len:
                        edge_features[i].append(0.0)
                    edge_features[i] = edge_features[i][:max_edge_len]
            
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(edge_features, dtype=torch.float)
            
            # Get global features
            global_feats = self.get_global_features(mol, smiles)
            
            return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, 
                       global_feats=global_feats)
            
        except Exception as e:
            print(f"Error processing SMILES {smiles}: {e}")
            return self.create_dummy_graph()
    
    def create_dummy_graph(self):
        """Create a dummy graph for failed SMILES"""
        node_features = torch.ones(1, 83)  # Adjusted feature dimension
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
        edge_attr = torch.ones(1, 18)  # Adjusted edge feature dimension
        global_feats = torch.zeros(30)  # Adjusted global feature dimension
        return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, 
                   global_feats=global_feats)


class AdvancedMolecularDataset(Dataset):
    """Enhanced dataset with efficient caching"""
    
    def __init__(self, smiles_list, targets=None, graph_converter=None, cache_graphs=True):
        self.smiles_list = smiles_list
        self.targets = targets
        self.graph_converter = graph_converter or AdvancedSMILESToGraph()
        self.cache_graphs = cache_graphs
        
        if cache_graphs:
            print("Pre-computing molecular graphs with enhanced features...")
            self.graphs = []
            failed_count = 0
            
            for i, smiles in enumerate(smiles_list):
                if i % 500 == 0 and i > 0:
                    print(f"Processed {i}/{len(smiles_list)} molecules ({failed_count} failed)")
                
                graph = self.graph_converter.smiles_to_graph(smiles)
                self.graphs.append(graph)
                
                if graph.x.shape[0] == 1 and torch.all(graph.x == 1):  # Dummy graph check
                    failed_count += 1
            
            print(f"Completed graph generation. Failed to parse {failed_count} SMILES strings")
        else:
            self.graphs = None
    
    def __len__(self):
        return len(self.smiles_list)
    
    def __getitem__(self, idx):
        if self.cache_graphs:
            graph = self.graphs[idx]
        else:
            graph = self.graph_converter.smiles_to_graph(self.smiles_list[idx])
        
        if self.targets is not None:
            target = torch.tensor(self.targets[idx], dtype=torch.float)
            return graph, target
        
        return graph


class TransformerGNN(nn.Module):
    """Advanced Transformer-based GNN with multi-head attention and hierarchical pooling"""
    
    def __init__(self, node_input_dim, edge_input_dim, global_input_dim, 
                 hidden_dim=512, output_dim=1, num_layers=8, num_heads=16, dropout=0.15):
        super(TransformerGNN, self).__init__()
        
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.dropout = dropout
        
        # Input projections with batch normalization
        self.node_proj = nn.Sequential(
            nn.Linear(node_input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        if edge_input_dim > 0:
            self.edge_proj = nn.Sequential(
                nn.Linear(edge_input_dim, hidden_dim // 4),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
        else:
            self.edge_proj = None
        
        self.global_proj = nn.Sequential(
            nn.Linear(global_input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Transformer-based graph convolutions
        self.transformer_convs = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        self.feed_forwards = nn.ModuleList()
        
        for i in range(num_layers):
            # Transformer convolution layer
            self.transformer_convs.append(
                TransformerConv(
                    hidden_dim, 
                    hidden_dim // num_heads, 
                    heads=num_heads,
                    dropout=dropout,
                    edge_dim=hidden_dim // 4 if self.edge_proj else None,
                    beta=True
                )
            )
            
            # Layer normalization
            self.layer_norms.append(LayerNorm(hidden_dim))
            
            # Feed-forward network
            self.feed_forwards.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 4, hidden_dim),
                nn.Dropout(dropout)
            ))
        
        # Multi-scale graph attention
        self.graph_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads // 2,
            dropout=dropout,
            batch_first=True
        )
        
        # Hierarchical pooling layers
        self.pool_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Multiple pooling strategies
        self.adaptive_pool = nn.AdaptiveMaxPool1d(1)
        
        # Feature fusion network
        pooled_dim = hidden_dim * 4  # mean, max, add, attention pooling
        
        self.feature_fusion = nn.Sequential(
            nn.Linear(pooled_dim + hidden_dim, hidden_dim * 2),  # +global features
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Final prediction head with residual connections
        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(dropout / 2),
            
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout / 4),
            
            nn.Linear(hidden_dim // 4, output_dim)
        )
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Initialize weights using Xavier/Kaiming initialization"""
        if isinstance(module, nn.Linear):
            torch.nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm1d):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        edge_attr = getattr(data, 'edge_attr', None)
        global_feats = getattr(data, 'global_feats', None)
        
        # Project node features
        x = self.node_proj(x)
        
        # Project edge features
        if edge_attr is not None and self.edge_proj is not None:
            edge_attr = self.edge_proj(edge_attr)
        
        # Apply transformer layers with residual connections
        for i in range(self.num_layers):
            # Transformer convolution
            x_new = self.transformer_convs[i](x, edge_index, edge_attr)
            
            # Residual connection and layer norm
            x = x + x_new
            x = self.layer_norms[i](x)
            
            # Feed-forward with residual connection
            x_ff = self.feed_forwards[i](x)
            x = x + x_ff
        
        # Multi-scale pooling
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x_add = global_add_pool(x, batch)
        
        # Attention-based pooling
        batch_size = batch.max().item() + 1
        x_att_pooled = []
        
        for i in range(batch_size):
            mask = (batch == i)
            if mask.sum() > 0:
                x_batch = x[mask].unsqueeze(0)  # [1, num_nodes, hidden_dim]
                
                # Self-attention for pooling
                att_out, _ = self.graph_attention(x_batch, x_batch, x_batch)
                
                # Attention weights for pooling
                att_weights = self.pool_attention(att_out.squeeze(0))  # [num_nodes, 1]
                att_weights = F.softmax(att_weights, dim=0)
                
                # Weighted sum
                x_att = torch.sum(att_weights * x[mask], dim=0)
                x_att_pooled.append(x_att)
            else:
                x_att_pooled.append(torch.zeros(self.hidden_dim).to(x.device))
        
        x_att = torch.stack(x_att_pooled)
        
        # Concatenate all pooled representations
        x_pooled = torch.cat([x_mean, x_max, x_add, x_att], dim=1)
        
        # Process global features
        if global_feats is not None:
            if len(global_feats.shape) == 1:
                global_feats = global_feats.unsqueeze(0).repeat(x_pooled.shape[0], 1)
            global_processed = self.global_proj(global_feats)
        else:
            global_processed = torch.zeros(x_pooled.shape[0], self.hidden_dim).to(x.device)
        
        # Feature fusion
        x_fused = torch.cat([x_pooled, global_processed], dim=1)
        x_fused = self.feature_fusion(x_fused)
        
        # Final prediction
        output = self.prediction_head(x_fused)
        
        return output


def train_advanced_model(train_loader, val_loader, node_dim, edge_dim, global_dim, 
                        target_name, epochs=200, patience=40):
    """Train advanced transformer-based GNN"""
    
    model = TransformerGNN(
        node_input_dim=node_dim,
        edge_input_dim=edge_dim,
        global_input_dim=global_dim,
        hidden_dim=512,
        output_dim=1,
        num_layers=8,
        num_heads=16,
        dropout=0.15
    ).to(device)
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Advanced optimizer with weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0005,
        weight_decay=1e-4,
        betas=(0.9, 0.999),
        eps=1e-8
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.001,
        total_steps=epochs * len(train_loader),
        pct_start=0.1,
        anneal_strategy='cos'
    )
    
    # Loss function - Huber loss for robustness
    criterion = nn.SmoothL1Loss(beta=1.0)
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    print(f"Training advanced model for {target_name}...")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_count = 0
        
        for batch_idx, (batch_data, batch_target) in enumerate(train_loader):
            batch_data = batch_data.to(device)
            batch_target = batch_target.to(device)
            
            optimizer.zero_grad()
            
            try:
                output = model(batch_data)
                loss = criterion(output.squeeze(), batch_target)
                
                # Gradient penalty for stability
                if epoch > 10:
                    grad_penalty = 0
                    for name, param in model.named_parameters():
                        if param.grad is not None:
                            grad_penalty += torch.norm(param.grad, p=2) ** 2
                    loss += 1e-6 * grad_penalty
                
                loss.backward()
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                
                optimizer.step()
                scheduler.step()
                
                train_loss += loss.item()
                train_count += 1
                
            except RuntimeError as e:
                print(f"Training error at batch {batch_idx}: {e}")
                continue
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_count = 0
        val_predictions = []
        val_actuals = []
        
        with torch.no_grad():
            for batch_data, batch_target in val_loader:
                batch_data = batch_data.to(device)
                batch_target = batch_target.to(device)
                
                try:
                    output = model(batch_data)
                    loss = criterion(output.squeeze(), batch_target)
                    
                    val_loss += loss.item()
                    val_count += 1
                    
                    val_predictions.extend(output.squeeze().cpu().numpy())
                    val_actuals.extend(batch_target.cpu().numpy())
                    
                except RuntimeError as e:
                    print(f"Validation error: {e}")
                    continue
        
        # Calculate metrics
        avg_train_loss = train_loss / max(train_count, 1)
        avg_val_loss = val_loss / max(val_count, 1)
        
        if len(val_predictions) > 0 and len(val_actuals) > 0:
            val_r2 = r2_score(val_actuals, val_predictions)
            val_mae = np.mean(np.abs(np.array(val_actuals) - np.array(val_predictions)))
        else:
            val_r2, val_mae = 0, 0
        
        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'epoch': epoch,
                'loss': best_val_loss,
            }, f'best_transformer_gnn_{target_name}.pth')
        else:
            patience_counter += 1
        
        if epoch % 20 == 0 or patience_counter == 0:
            print(f'Epoch {epoch:03d}: Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}, Val R²: {val_r2:.3f}, '
                  f'Val MAE: {val_mae:.4f}, LR: {scheduler.get_last_lr()[0]:.2e}')
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}')
            break
    
    # Load best model
    try:
        checkpoint = torch.load(f'best_transformer_gnn_{target_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
    except:
        print("Warning: Could not load best model, using current state")
    
    return model


def advanced_collate_fn(batch):
    """Advanced collate function with error handling"""
    try:
        if len(batch[0]) == 2:  # Training data
            graphs, targets = zip(*batch)
            targets = torch.stack(targets)
        else:  # Test data
            graphs = batch
            targets = None
        
        # Handle global features
        batch_graphs = []
        global_feats_list = []
        
        for graph in graphs:
            if hasattr(graph, 'global_feats') and graph.global_feats is not None:
                global_feats_list.append(graph.global_feats)
            else:
                global_feats_list.append(torch.zeros(30))  # Default size
            
            # Create graph without global features for batching
            batch_graph = Data(
                x=graph.x,
                edge_index=graph.edge_index,
                edge_attr=getattr(graph, 'edge_attr', None)
            )
            batch_graphs.append(batch_graph)
        
        # Batch graphs
        batch_data = Batch.from_data_list(batch_graphs)
        
        # Add global features back
        if global_feats_list:
            batch_data.global_feats = torch.stack(global_feats_list)
        
        if targets is not None:
            return batch_data, targets
        return batch_data
        
    except Exception as e:
        print(f"Collate function error: {e}")
        # Return dummy batch
        dummy_graph = Data(
            x=torch.ones(1, 83),
            edge_index=torch.tensor([[0], [0]], dtype=torch.long),
            edge_attr=torch.ones(1, 18),
            global_feats=torch.zeros(30)
        )
        if len(batch[0]) == 2:
            return dummy_graph, torch.zeros(1)
        return dummy_graph


# Initialize advanced graph converter
print("Initializing advanced molecular graph converter...")
graph_converter = AdvancedSMILESToGraph()

# Create datasets
print("Creating advanced molecular datasets...")

# Test dataset
test_dataset = AdvancedMolecularDataset(
    test['SMILES'].tolist(),
    graph_converter=graph_converter,
    cache_graphs=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,  # Smaller batch size for stability
    shuffle=False,
    collate_fn=advanced_collate_fn,
    num_workers=0
)

# Get feature dimensions
sample_graph = test_dataset[0]
node_dim = sample_graph.x.shape[1]
edge_dim = sample_graph.edge_attr.shape[1] if hasattr(sample_graph, 'edge_attr') else 0
global_dim = sample_graph.global_feats.shape[0] if hasattr(sample_graph, 'global_feats') else 0

print(f"Advanced feature dimensions:")
print(f"  Node features: {node_dim}")
print(f"  Edge features: {edge_dim}")
print(f"  Global features: {global_dim}")

# Training and prediction
target_columns = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
predictions = {}
models = {}

for target in target_columns:
    print(f"\n{'='*60}")
    print(f"Training advanced model for {target}")
    print(f"{'='*60}")
    
    # Prepare target data
    target_train = train[['SMILES', target]].dropna()
    
    if len(target_train) < 50:
        print(f"Skipping {target} - insufficient data ({len(target_train)} samples)")
        continue
    
    print(f"Training samples: {len(target_train)}")
    
    # Create training dataset
    train_dataset = AdvancedMolecularDataset(
        target_train['SMILES'].tolist(),
        target_train[target].values,
        graph_converter=graph_converter,
        cache_graphs=True
    )
    
    # Split data
    train_indices, val_indices = train_test_split(
        range(len(train_dataset)),
        test_size=0.15,
        random_state=42
    )
    
    train_subset = torch.utils.data.Subset(train_dataset, train_indices)
    val_subset = torch.utils.data.Subset(train_dataset, val_indices)
    
    # Create data loaders
    train_loader = DataLoader(
        train_subset,
        batch_size=16,
        shuffle=True,
        collate_fn=advanced_collate_fn,
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_subset,
        batch_size=16,
        shuffle=False,
        collate_fn=advanced_collate_fn,
        num_workers=0
    )
    
    print(f"Training batches: {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")
    
    # Train model
    try:
        model = train_advanced_model(
            train_loader, val_loader, node_dim, edge_dim, global_dim,
            target, epochs=150, patience=30
        )
        
        models[target] = model
        
        # Make predictions
        model.eval()
        test_predictions = []
        
        print("Making predictions on test set...")
        with torch.no_grad():
            for batch_data in test_loader:
                batch_data = batch_data.to(device)
                try:
                    output = model(batch_data)
                    test_predictions.extend(output.squeeze().cpu().numpy())
                except Exception as e:
                    print(f"Prediction error: {e}")
                    # Add dummy predictions for failed batches
                    batch_size = batch_data.batch.max().item() + 1
                    test_predictions.extend([0.0] * batch_size)
        
        predictions[target] = test_predictions
        print(f"Generated {len(test_predictions)} predictions for {target}")
        
    except Exception as e:
        print(f"Training failed for {target}: {e}")
        # Use mean value as fallback
        mean_val = target_train[target].mean()
        predictions[target] = [mean_val] * len(test)
        print(f"Using fallback mean value: {mean_val:.3f}")


# Create submission
print("\n" + "="*50)
print("Creating final submission...")
print("="*50)

submission_data = {'id': test['id']}

for target in target_columns:
    if target in predictions:
        # Ensure correct length
        pred_values = predictions[target]
        if len(pred_values) != len(test):
            print(f"Warning: {target} predictions length mismatch. Expected {len(test)}, got {len(pred_values)}")
            if len(pred_values) > len(test):
                pred_values = pred_values[:len(test)]
            else:
                mean_val = np.mean(pred_values) if pred_values else 0.0
                pred_values.extend([mean_val] * (len(test) - len(pred_values)))
        
        submission_data[target] = pred_values
    else:
        # Use training mean as fallback
        if target in train.columns and not train[target].isna().all():
            mean_val = train[target].mean()
        else:
            mean_val = 0.0
        submission_data[target] = [mean_val] * len(test)
        print(f"Using fallback for {target}: {mean_val:.3f}")

# Create DataFrame and apply bounds
submission_df = pd.DataFrame(submission_data)

# Apply reasonable bounds based on training data
for target in target_columns:
    if target in train.columns and not train[target].isna().all():
        q01 = train[target].quantile(0.005)
        q99 = train[target].quantile(0.995)
        submission_df[target] = np.clip(submission_df[target], q01, q99)

# Save submission
submission_df.to_csv('submission.csv', index=False)
print(f"Submission saved as 'submission.csv'")
print(f"Shape: {submission_df.shape}")

# Display statistics
print("\n" + "="*40)
print("SUBMISSION STATISTICS")
print("="*40)

for target in target_columns:
    values = submission_df[target].values
    print(f"\n{target}:")
    print(f"  Mean: {values.mean():.4f} ± {values.std():.4f}")
    print(f"  Range: [{values.min():.4f}, {values.max():.4f}]")
    print(f"  Median: {np.median(values):.4f}")

print("\nFirst 5 predictions:")
print(submission_df.head())

print("\n" + "="*50)
print("ADVANCED GNN TRAINING COMPLETED!")
print("="*50)
print("Key improvements:")
print("✓ Comprehensive atom/bond feature extraction (80+ features)")
print("✓ Transformer-based GNN architecture with multi-head attention")
print("✓ Advanced global molecular descriptors (30+ features)")
print("✓ Hierarchical pooling with attention mechanism")
print("✓ Robust training with gradient clipping and regularization")
print("✓ Error handling and fallback mechanisms")
print("✓ Optimized for polymer property prediction")

Using device: cuda
Initializing advanced molecular graph converter...
Creating advanced molecular datasets...
Pre-computing molecular graphs with enhanced features...
Completed graph generation. Failed to parse 0 SMILES strings
Advanced feature dimensions:
  Node features: 94
  Edge features: 18
  Global features: 28

Training advanced model for Tg
Training samples: 511
Pre-computing molecular graphs with enhanced features...
Processed 500/511 molecules (0 failed)
Completed graph generation. Failed to parse 0 SMILES strings
Training batches: 28
Validation batches: 5
Model parameters: 30,444,418
Training advanced model for Tg...
Epoch 000: Train Loss: 111.8463, Val Loss: 94.2758, Val R²: -0.666, Val MAE: 94.4814, LR: 5.05e-05
Epoch 002: Train Loss: 107.2015, Val Loss: 92.4067, Val R²: -0.590, Val MAE: 92.6382, LR: 1.32e-04
Epoch 003: Train Loss: 110.1104, Val Loss: 88.0662, Val R²: -0.457, Val MAE: 88.2824, LR: 2.00e-04
Epoch 004: Train Loss: 104.3548, Val Loss: 85.8251, Val R²: -0.399,